# Lab 2: ทำความสะอาดข้อมูลดิบจากเซนเซอร์ IoT

ขั้นตอน: ลบข้อมูลซ้ำ · แปลงค่าผิดปกติเป็น NULL · จัดการ Missing Value · เพิ่มคอลัมน์ `data_quality_status` · สรุปจำนวน Error / Duplicate / Missing / Outlier

## 1) โหลด Raw Data (เก็บต้นฉบับไว้ ไม่แก้ไข)

In [1]:
import pandas as pd
import numpy as np

RAW_FILE = "iot_sensor_raw_data_extended.csv"
CLEAN_FILE = "iot_sensor_cleaned_data.csv"
NA_TOKENS = ["NULL", "null", "", "NaN", "nan", "None"]

# โหลดเป็น string ทั้งหมดก่อน เพื่อคุมการแปลงค่าเอง
raw = pd.read_csv(RAW_FILE, dtype=str, keep_default_na=False)
n_before = len(raw)
df = raw.copy()
report = {"missing": 0, "duplicate": 0, "outlier": 0, "error": 0}
print("จำนวนแถว Raw:", n_before)
raw.head()

จำนวนแถว Raw: 480


,timestamp,device_id,temp_c,motion,battery
0,2026-07-12 14:50:00,SN-9982,28.3,false,88
1,2026-07-12 14:50:00,SN-9983,28.2,true,94
2,2026-07-12 14:50:00,SN-9984,29.7,true,76
3,2026-07-12 14:50:00,SN-9985,27.8,true,81
4,2026-07-12 14:50:00,SN-9986,30.7,true,69


## 2) Data Quality Rules
- `temp_c` ต้องอยู่ 0–50 °C (`-999` = sensor error, นอกช่วง = outlier)
- `device_id` ห้ามว่าง
- `battery` ต้องอยู่ 0–100
- `motion` ต้องเป็น true/false
- ห้ามซ้ำที่ `device_id` + `timestamp` เดียวกัน

### 2.1 แปลง token ค่าว่าง → NaN

In [2]:
df = df.replace(NA_TOKENS, np.nan)

### 2.2 Normalize คอลัมน์ motion (ค่าผิดรูป → NULL)

In [3]:
def norm_motion(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in ("true", "t", "1", "yes"):
        return "true"
    if s in ("false", "f", "0", "no"):
        return "false"
    return "__ERROR__"   # FALSEE, TRUEE, unknown

df["motion"] = df["motion"].apply(norm_motion)
motion_error_mask = df["motion"] == "__ERROR__"
report["error"] += int(motion_error_mask.sum())
df.loc[motion_error_mask, "motion"] = np.nan
print("motion ผิดรูป (error):", int(motion_error_mask.sum()))

motion ผิดรูป (error): 3


### 2.3 แปลง temp_c / battery เป็นตัวเลข แล้วจัดการ Error/Outlier → NULL

In [4]:
df["temp_c"] = pd.to_numeric(df["temp_c"], errors="coerce")
df["battery"] = pd.to_numeric(df["battery"], errors="coerce")

# temp_c: -999 = sensor error, นอกช่วง 0-50 = outlier
temp_error_mask = df["temp_c"] <= -900
temp_outlier_mask = (~temp_error_mask) & ((df["temp_c"] < 0) | (df["temp_c"] > 50)) & df["temp_c"].notna()
report["error"] += int(temp_error_mask.sum())
report["outlier"] += int(temp_outlier_mask.sum())
df.loc[temp_error_mask | temp_outlier_mask, "temp_c"] = np.nan

# battery: นอกช่วง 0-100 = outlier
batt_outlier_mask = ((df["battery"] < 0) | (df["battery"] > 100)) & df["battery"].notna()
report["outlier"] += int(batt_outlier_mask.sum())
df.loc[batt_outlier_mask, "battery"] = np.nan

print("temp error(-999):", int(temp_error_mask.sum()),
      "| temp outlier:", int(temp_outlier_mask.sum()),
      "| battery outlier:", int(batt_outlier_mask.sum()))

temp error(-999): 6 | temp outlier: 4 | battery outlier: 5


## 3) ลบข้อมูลซ้ำ (device_id + timestamp)

In [5]:
dup_mask = df.duplicated(subset=["device_id", "timestamp"], keep="first")
report["duplicate"] = int(dup_mask.sum())
df = df[~dup_mask].copy()
print("แถวซ้ำที่ลบ:", report["duplicate"])

แถวซ้ำที่ลบ: 25


## 4) นับ Missing Value (หลังแปลง error/outlier เป็น NULL)

In [6]:
report["missing"] = int(df[["device_id", "temp_c", "motion", "battery"]].isna().sum().sum())
df[["device_id", "temp_c", "motion", "battery"]].isna().sum()

device_id     2
temp_c       15
motion        7
battery       8
dtype: int64

## 5) เพิ่มคอลัมน์ data_quality_status
- `OK` = ครบทุกค่า
- `INVALID_NO_DEVICE_ID` = device_id ว่าง
- `MISSING_VALUE` = มีค่าว่างในคอลัมน์อื่น

In [7]:
def quality_status(row):
    if pd.isna(row["device_id"]):
        return "INVALID_NO_DEVICE_ID"
    if row[["temp_c", "motion", "battery"]].isna().any():
        return "MISSING_VALUE"
    return "OK"

df["data_quality_status"] = df.apply(quality_status, axis=1)
df["data_quality_status"].value_counts()

data_quality_status
OK                      423
MISSING_VALUE            30
INVALID_NO_DEVICE_ID      2
Name: count, dtype: int64

## 6) บันทึกไฟล์ Cleaned Data

In [8]:
n_after = len(df)
df.to_csv(CLEAN_FILE, index=False)
print("บันทึก:", CLEAN_FILE, "| แถวหลัง clean:", n_after)
df.head()

บันทึก: iot_sensor_cleaned_data.csv | แถวหลัง clean: 455


,timestamp,device_id,temp_c,motion,battery,data_quality_status
0,2026-07-12 14:50:00,SN-9982,28.3,false,88.0,OK
1,2026-07-12 14:50:00,SN-9983,28.2,true,94.0,OK
2,2026-07-12 14:50:00,SN-9984,29.7,true,76.0,OK
3,2026-07-12 14:50:00,SN-9985,27.8,true,81.0,OK
4,2026-07-12 14:50:00,SN-9986,30.7,true,69.0,OK


## 7) รายงานสรุป

In [9]:
print("=" * 55)
print(" สรุปผลการทำความสะอาดข้อมูล IoT Sensor")
print("=" * 55)
print(f"จำนวนแถวก่อนทำความสะอาด : {n_before}")
print(f"จำนวนแถวหลังทำความสะอาด : {n_after}")
print(f"แถวที่ถูกลบ (ซ้ำ)       : {n_before - n_after}")
print("-" * 55)
print(f"Error (ค่าผิด/เซนเซอร์เสีย) : {report['error']}")
print(f"Duplicate (ข้อมูลซ้ำ)      : {report['duplicate']}")
print(f"Missing Value (ค่าว่าง)    : {report['missing']}")
print(f"Outlier (นอกช่วงที่กำหนด)  : {report['outlier']}")
print("-" * 55)
summary = pd.DataFrame({
    "ประเภทปัญหา": ["Error", "Duplicate", "Missing Value", "Outlier"],
    "จำนวน": [report["error"], report["duplicate"], report["missing"], report["outlier"]],
})
summary

 สรุปผลการทำความสะอาดข้อมูล IoT Sensor
จำนวนแถวก่อนทำความสะอาด : 480
จำนวนแถวหลังทำความสะอาด : 455
แถวที่ถูกลบ (ซ้ำ)       : 25
-------------------------------------------------------
Error (ค่าผิด/เซนเซอร์เสีย) : 9
Duplicate (ข้อมูลซ้ำ)      : 25
Missing Value (ค่าว่าง)    : 32
Outlier (นอกช่วงที่กำหนด)  : 9
-------------------------------------------------------


,ประเภทปัญหา,จำนวน
0,Error,9
1,Duplicate,25
2,Missing Value,32
3,Outlier,9
